# **Interactive exploration of the Amundsen Gulf CTD transect**


*Surface hydrographic properties and sampling stations along the cruise route*

For an interactive version of this page please visit the Google Colab: [Open in Google Colab](https://colab.research.google.com/drive/1dbGTSRUqcDaY_frLyqe2MsR7tL6IzwDQ#scrollTo=hK0fR-TgrXzR)

*(To open link in new tab press Ctrl + click)*

Alternatevly this notebook can be opened with Binder by following the link: [Interactive exploration of the Amundsen Gulf CTD transect](https://mybinder.org/v2/gh/s4oceanice/literacy.s4oceanice/main?urlpath=%2Fdoc%2Ftree%2Fnotebooks_binder%2FAMUNDSEN_route.ipynb)

**Scientific background**

CTD observations provide information on the physical structure of the ocean by measuring conductivity, temperature, and pressure throughout the water column. Derived variables such as salinity and density support the interpretation of water masses, mixing processes, and spatial gradients.

The Amundsen Gulf is an important Arctic marine environment influenced by seasonal sea ice, freshwater inputs, atmospheric forcing, and exchanges with adjacent shelf and basin waters.


**Cruise overview**

This notebook provides an interactive environment to explore CTD (Conductivity-Temperature-Depth) cast data collected during the Amundsen Gulf cruise (CCIN 1498, CFL 0802) distribuited through the EMODnet Physics ERDDAP service. The interactive map displays the cruise stations and allows users to colour the sampling locations according to the available numerical parameters.

**Purpose**

This notebook enables users to:
*   Select an oceanographic variable (e.g., temperature, salinity, dissolved
oxygen) for visualization.
*   View the station track on an interactive map, with each station colored according to the selected variable's surface value.
*   Explore individual station popups showing the variable's value and unit.

**Data sources**

The notebook uses the following dataset: https://erddap.emodnet-physics.eu/erddap/tabledap/ARICE_CCIN1498_CFL0802_AmundsenGulf.html

This dataset consists of CTD casts: at each station, a sensor package is lowered through the water column, producing multiple measurements at different depths. This notebook displays a single, surface (shallowest-depth) value per station on the map, so that each station appears as one point rather than several overlapping ones from different depths.

Note: variable names and available parameters should be verified directly against this dataset's ERDDAP metadata, as they have not yet been individually confirmed for this cruise.

**How to use this notebook**

1. Run the code cells sequentially from top to bottom.
2. Wait for the remote dataset to be downloaded and processed.
3. Use the available menus, sliders, or map controls to select the variables and periods of interest.
4. Read the interpretation guidance before drawing scientific conclusions from the visualizations.

The notebook retrieves data from remote services. An active internet connection is therefore required.

**Data retrieval and preparation**

The code below imports the required libraries, retrieves the dataset, prepares the station records, identifies the numerical variables available for mapping, and generates the interactive `folium` visualization.

**Interactive visualization**

Use the parameter selector to choose the variable represented by the station colours.

- Marker position represents the station coordinates.
- Marker colour represents the selected measurement.
- The colour scale reports the corresponding range of values.
- Popups provide station-specific information.
- The connecting line gives an approximate representation of the station sequence.

In [ ]:
# @title
import pandas as pd
import folium
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from IPython.display import display
import ipywidgets as widgets
import numpy as np

# Dataset URL
DATA_URL = 'https://erddap.emodnet-physics.eu/erddap/tabledap/ARICE_CCIN1498_CFL0802_AmundsenGulf.csv'

def load_and_preprocess_data(url):
    """Loads CTD data and keeps only the surface (shallowest-depth) value per station."""
    try:
        df = pd.read_csv(url, skiprows=[1])

        lat_col = [col for col in df.columns if 'latitude' in col.lower()][0]
        lon_col = [col for col in df.columns if 'longitude' in col.lower()][0]
        df = df.rename(columns={lat_col: 'latitude', lon_col: 'longitude'})

        df = df.dropna(subset=['latitude', 'longitude'])

        for col in df.columns:
            if 'time' not in col.lower():
                converted = pd.to_numeric(df[col], errors='coerce')
                if not converted.isna().all():
                    df[col] = converted

        depth_cols = [col for col in df.columns if 'depth' in col.lower()]
        if depth_cols:
            depth_col = depth_cols[0]
            time_col = [col for col in df.columns if 'time' in col.lower()]
            group_cols = ['latitude', 'longitude'] + (time_col[:1] if time_col else [])
            df = df.sort_values(depth_col).groupby(group_cols, as_index=False).first()

        return df
    except Exception as e:
        print(f"Error loading data from {url}: {e}")
        return pd.DataFrame()

df_amundsen = load_and_preprocess_data(DATA_URL)
print("Data loading complete.")

# Variable metadata: human-readable names and units
# NOTE: confirm variable codes against this dataset's .das metadata before relying on this list
VARIABLE_METADATA = {
    "depth": ("Depth", "m"),
    "TEMP": ("Sea water temperature", "°C"),
    "PSAL": ("Practical salinity", "PSU"),
    "DENS": ("Sea density [sigma-theta]", "kg/m3"),
    "DOX1": ("Dissolved oxygen", "ml/l"),
}

def get_variable_label(variable_name):
    display_name, unit = VARIABLE_METADATA.get(variable_name, (variable_name, ""))
    return f"{display_name} ({unit})" if unit else display_name

def plot_colored_route(variable_name):
    if variable_name not in df_amundsen.columns:
        print(f"Variable not found: {variable_name}")
        return

    df = df_amundsen.dropna(subset=["latitude", "longitude", variable_name])
    if df.empty:
        print("No valid data available for this variable.")
        return

    display_name, unit = VARIABLE_METADATA.get(variable_name, (variable_name, ""))
    label = get_variable_label(variable_name)

    m = folium.Map(location=[df["latitude"].mean(), df["longitude"].mean()], zoom_start=6)

    vmin, vmax = df[variable_name].min(), df[variable_name].max()
    colormap = plt.get_cmap("viridis")
    norm = colors.Normalize(vmin=vmin, vmax=vmax) if vmin != vmax else colors.Normalize(vmin=vmin - 1, vmax=vmax + 1)

    for _, row in df.iterrows():
        val = row[variable_name]
        hex_color = colors.rgb2hex(colormap(norm(val))[:3])
        popup_text = f"<b>{display_name}</b><br>{val:.2f} {unit}" if unit else f"<b>{display_name}</b><br>{val:.2f}"
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=5, color=hex_color, fill=True, fill_color=hex_color,
            fill_opacity=0.8, popup=folium.Popup(popup_text, max_width=300)
        ).add_to(m)

    display(m)

    fig, ax = plt.subplots(figsize=(8, 1))
    fig.subplots_adjust(bottom=0.5)
    fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=colormap), cax=ax, orientation="horizontal", label=label)
    plt.show()

numeric_cols = df_amundsen.select_dtypes(include=[np.number]).columns.tolist()
exclude_keywords = ["latitude", "longitude", "time", "depth"]
vars_available = [c for c in numeric_cols if not any(x in c.lower() for x in exclude_keywords) and not c.upper().endswith("_QC")]

variable_dropdown = widgets.Dropdown(
    options=[(get_variable_label(c), c) for c in vars_available],
    description="Variable:"
)
output_map = widgets.Output()

def on_change(change):
    with output_map:
        output_map.clear_output(wait=True)
        if variable_dropdown.value:
            plot_colored_route(variable_dropdown.value)

variable_dropdown.observe(on_change, "value")
display(variable_dropdown, output_map)
on_change(None)

Data loading complete.


Dropdown(description='Variable:', options=(('Sea water temperature (°C)', 'TEMP'), ('TUR3', 'TUR3'), ('FLU2', …

Output()

**Interpretation guidance**

The map is intended for exploratory spatial analysis. The displayed values represent the shallowest valid observation retained for each station and do not describe the complete vertical structure of the water column.

Apparent spatial gradients should be evaluated together with sampling time, station spacing, depth, units, and data availability. The connecting line follows the order used by the processed table and should not be interpreted as a high-resolution vessel trajectory.


**Additional resources and acknowledgement**

Libraries used:
*  [pandas](https://pandas.pydata.org/) for data handling
*  numpy for data handling
*  folium for interactive mapping
*  [matplotlib](https://matplotlib.org/) for colour normalization
*  [ipywidgets](https://ipywidgets.readthedocs.io/en/stable/) foir the parameter selector.


This work has received funding from the European Union Horizon Europe project Ocean-Cryosphere Exchanges in ANtarctica: Impacts on Climate and the Earth System (OCEAN ICE) under grant agreement No. 101060452 (https://doi.org/10.3030/101060452). UK partners are funded by UK Research and Innovation (UKRI) under the UK government's Horizon Europe funding guarantee.

This notebook makes use of data from the ARICE (Arctic Research Icebreaker Consortium) project, hosted via EMODnet Physics (https://www.emodnet-physics.eu).

<center>
  <div style="display: flex; justify-content: center; align-items: flex-start; gap: 20px;">
    <img src="https://ocean-ice.eu/wp-content/uploads/2025/02/TO-USE-RGB-for-digital-materials-V.png" height="140" style="margin-top: 50px;"/>
    <img src="https://ocean-ice.eu/wp-content/uploads/2025/02/UKRI-logo-1.png" height="100"/>
    <img src="https://ocean-ice.eu/wp-content/uploads/2023/06/logo-polar-cluster-2.png" height="100"/>
    <img src="https://emodnet.ec.europa.eu/sites/emodnet.ec.europa.eu/files/public/emodnet_logos/web/EMODnet_standard_colour.png" height="100"/>
  </div>
</center>